# 01 - Data Extraction

This notebook handles the initial data loading step of the ETL pipeline.
It auto-detects the file format, loads the raw dataset, performs basic inspection,
and samples exactly 100,000 rows for use in downstream notebooks.

After sampling, `student_id` is reassigned as a sequential integer (1 to N)
so that IDs remain consistent and gap-free throughout the pipeline.

## 1. Imports and Configuration

In [31]:
import os
import pathlib
import pandas as pd
import numpy as np

# Seed for reproducibility
RANDOM_STATE = 42
SAMPLE_SIZE  = 100_000

# Path to raw data folder
RAW_DIR = pathlib.Path(os.path.join("..", "data", "raw"))
print(f"Raw data directory : {RAW_DIR.resolve()}")
print(f"Exists             : {RAW_DIR.exists()}")

Raw data directory : /Users/soumyatiwari/Desktop/B_G6_Digitomics/data/raw
Exists             : True


## 2. Auto-Detect and Load Raw File

The loader scans `data/raw/` for the first supported file (CSV, Excel, JSON)
and reads it into a pandas DataFrame. Priority order: CSV > Excel > JSON.

In [32]:
def load_raw_file(directory: pathlib.Path) -> pd.DataFrame:
    """Auto-detect and load the first supported file in the given directory."""
    supported = {
        ".csv":  lambda p: pd.read_csv(p, low_memory=False),
        ".xlsx": lambda p: pd.read_excel(p),
        ".xls":  lambda p: pd.read_excel(p),
        ".json": lambda p: pd.read_json(p),
    }

    for filepath in sorted(directory.iterdir()):
        ext = filepath.suffix.lower()
        if ext in supported and not filepath.name.startswith("."):
            print(f"Detected file : {filepath.name}")
            print(f"Format        : {ext.upper().lstrip('.')}")
            df = supported[ext](filepath)
            print(f"Loaded        : {len(df):,} rows x {len(df.columns)} columns")
            return df

    raise FileNotFoundError(f"No supported file found in {directory}")


df_raw = load_raw_file(RAW_DIR)

Detected file : student_digital_behaviour_data.csv
Format        : CSV
Loaded        : 500,000 rows x 48 columns


## 3. Basic Dataset Inspection

A quick overview of the dataset: shape, column types, and a preview of the first rows.

In [33]:
# Shape
print("=" * 50)
print(f"Shape : {df_raw.shape[0]:,} rows, {df_raw.shape[1]} columns")
print("=" * 50)

Shape : 500,000 rows, 48 columns


In [34]:
# Column data types
print("Column dtypes:")
print(df_raw.dtypes)

Column dtypes:
student_id                          int64
country                            object
development_level                  object
poverty_rate_percent              float64
internet_infrastructure_index     float64
average_internet_speed_mbps       float64
age                                 int64
gender                             object
urban_rural                        object
family_income_level                object
device_access                      object
internet_access_hours             float64
education_level                    object
field_of_study                     object
academic_motivation               float64
online_learning_hours             float64
social_media_hours                float64
sessions_per_day                  float64
average_session_length_minutes    float64
late_night_usage                   object
education_content_hours           float64
short_video_hours                 float64
entertainment_content_hours       float64
news_content_hours 

In [35]:
# First 5 rows
print("First 5 rows:")
df_raw.head()

First 5 rows:


,student_id,country,development_level,poverty_rate_percent,internet_infrastructure_index,average_internet_speed_mbps,age,gender,urban_rural,family_income_level,...,ads_viewed_per_day,ads_clicked_per_week,impulse_purchase_score,digital_spending_per_month,cyberbullying_exposure,adult_content_exposure,digital_addiction_score,wellbeing_index,academic_risk_score,financial_risk_score
0,1,Qatar,Developing,16.30,54.93,27.19,21,Male,Rural,High,...,38.699484,3.119569,3.681066,79.508194,No,No,8.179932,66.662166,0.0,26.516722
1,2,USA,Developed,8.75,94.39,85.34,25,Male,Urban,Middle,...,157.400429,18.358090,6.538867,73.452464,No,Yes,22.073122,43.892278,0.0,39.257741
2,3,Mexico,Developing,23.64,47.24,73.55,18,Female,Urban,Low,...,79.603536,11.758299,5.150660,35.753069,No,No,12.591830,65.484625,0.0,47.542155
3,4,Canada,Developed,14.51,90.50,188.19,25,Male,Urban,Middle,...,69.318324,7.906197,3.195383,47.607487,No,Yes,8.008238,57.909392,0.0,23.436122
4,5,Sri Lanka,Underdeveloped,62.28,36.84,11.02,15,Other,Rural,Middle,...,144.019899,19.427839,7.180234,82.941313,No,No,21.551334,53.686356,0.0,47.308493


In [36]:
# Concise summary: non-null counts and dtype per column
print("Dataset info:")
df_raw.info(show_counts=True)

Dataset info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 48 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   student_id                      500000 non-null  int64  
 1   country                         500000 non-null  object 
 2   development_level               500000 non-null  object 
 3   poverty_rate_percent            500000 non-null  float64
 4   internet_infrastructure_index   500000 non-null  float64
 5   average_internet_speed_mbps     500000 non-null  float64
 6   age                             500000 non-null  int64  
 7   gender                          500000 non-null  object 
 8   urban_rural                     500000 non-null  object 
 9   family_income_level             500000 non-null  object 
 10  device_access                   500000 non-null  object 
 11  internet_access_hours           500000 non-null  float64
 12  ed

## 4. Missing Value Overview

A high-level look at null counts before any cleaning is performed.

In [37]:
null_summary = pd.DataFrame({
    "null_count":   df_raw.isnull().sum(),
    "null_percent": (df_raw.isnull().mean() * 100).round(2)
})

# Show only columns that have at least one null
null_summary = null_summary[null_summary["null_count"] > 0]
if null_summary.empty:
    print("No null values detected in the raw dataset.")
else:
    print(null_summary.sort_values("null_percent", ascending=False))

                 null_count  null_percent
field_of_study       247349         49.47
brain_rot_level        6262          1.25


## 5. Sampling 100,000 Rows

If the dataset exceeds 100,000 rows, a random sample of exactly that size is drawn
using `random_state=42` so results are reproducible. If the dataset is smaller,
all rows are kept and a notice is printed.

After sampling, `student_id` is reassigned as a sequential integer (1 to N).
The original IDs from the raw file are no longer consecutive once rows are
randomly shuffled, so reassignment ensures IDs are consistent and gap-free
in all downstream notebooks.

In [38]:
total_rows = len(df_raw)

if total_rows > SAMPLE_SIZE:
    df_sample = df_raw.sample(n=SAMPLE_SIZE, random_state=RANDOM_STATE).reset_index(drop=True)
    dropped   = total_rows - SAMPLE_SIZE
    print(f"Total rows in raw dataset : {total_rows:,}")
    print(f"Rows sampled              : {len(df_sample):,}")
    print(f"Rows dropped (not sampled): {dropped:,}")
    print(f"Sampling fraction         : {SAMPLE_SIZE / total_rows * 100:.2f}%")
else:
    df_sample = df_raw.copy().reset_index(drop=True)
    print(f"Dataset has {total_rows:,} rows (less than {SAMPLE_SIZE:,}). Using all rows.")
    print("Rows dropped: 0")

# Reassign student_id to sequential integers (1 to N) after random sampling.
# Random sampling scrambles the original IDs, making them non-contiguous.
# Reassigning produces a clean, gap-free ID column for downstream use.
if "student_id" in df_sample.columns:
    df_sample["student_id"] = range(1, len(df_sample) + 1)
    print(f"student_id reassigned: 1 to {len(df_sample):,}")
else:
    print("No student_id column found - skipping ID reassignment.")

Total rows in raw dataset : 500,000
Rows sampled              : 100,000
Rows dropped (not sampled): 400,000
Sampling fraction         : 20.00%
student_id reassigned: 1 to 100,000


In [39]:
# Confirm sample shape and verify student_id range
print(f"Sample shape: {df_sample.shape}")
if "student_id" in df_sample.columns:
    print(f"student_id range: {df_sample['student_id'].min()} to {df_sample['student_id'].max()}")
df_sample.head()

Sample shape: (100000, 48)
student_id range: 1 to 100000


,student_id,country,development_level,poverty_rate_percent,internet_infrastructure_index,average_internet_speed_mbps,age,gender,urban_rural,family_income_level,...,ads_viewed_per_day,ads_clicked_per_week,impulse_purchase_score,digital_spending_per_month,cyberbullying_exposure,adult_content_exposure,digital_addiction_score,wellbeing_index,academic_risk_score,financial_risk_score
0,1,Canada,Developed,14.51,90.50,188.19,15,Female,Urban,High,...,139.742376,13.717203,5.393212,106.593080,No,No,17.143210,54.673310,0.0,36.330249
1,2,Vietnam,Developing,30.70,66.87,45.63,21,Female,Urban,Middle,...,122.072394,14.108032,7.362823,82.212382,No,No,19.323299,49.714048,0.0,53.896830
2,3,Netherlands,Developed,14.70,80.43,107.41,17,Female,Urban,Low,...,133.576406,18.368277,6.724564,26.585072,No,No,15.678346,51.468265,0.0,51.876555
3,4,Argentina,Developing,18.99,68.14,69.08,18,Female,Rural,Middle,...,133.290182,17.051713,7.782861,97.834418,No,Yes,20.237128,59.181903,0.0,51.144355
4,5,Vietnam,Developing,30.70,66.87,45.63,21,Female,Rural,Low,...,88.523393,11.634842,5.258312,37.778477,No,No,8.286238,71.058998,0.0,46.642992


## 6. Store Sample for Downstream Notebooks

The sampled DataFrame is persisted using IPython's `%store` magic so that
notebook 02 can retrieve it without re-running this notebook.

In [40]:
# Persist the sampled dataframe for use in 02_cleaning.ipynb
%store df_sample
print("df_sample stored successfully.")

Stored 'df_sample' (DataFrame)
df_sample stored successfully.


## Summary

- Raw file auto-detected and loaded from `data/raw/`.
- Dataset inspected: shape, dtypes, first 5 rows, and null overview.
- Exactly 100,000 rows sampled with `random_state=42`.
- `student_id` reassigned as sequential integers (1 to 100,000) to ensure consistency after shuffling.
- `df_sample` stored for use in the next notebook (`02_cleaning.ipynb`).
- No data has been saved to disk in this step.